# CompileML deploy targets — one artifact, identical integers

The artifact built in the quickstart runs as pure-stdlib Python, as one SQL
query, and as a COBOL program. This notebook generates the SQL export,
**executes it in SQLite**, and proves per-row integer equality with the Python
runtime — then generates the COBOL and shows what a mainframe team receives.

In [1]:
import numpy as np

def make_credit_data(n=30_000, seed=42):
    """Synthetic credit-style dataset with a known data-generating process."""
    rng = np.random.default_rng(seed)
    utilization      = rng.beta(2, 4, n) * 1.2            # can exceed 1.0
    payment_ratio    = rng.beta(5, 2, n)                  # share of balance paid
    bills_paid_late  = rng.poisson(0.8, n).astype(float)  # late payments, 6m
    months_on_book   = rng.gamma(6, 8, n)
    n_credit_lines   = rng.poisson(4, n).astype(float) + 1
    inquiries_6m     = rng.poisson(1.2, n).astype(float)
    balance_to_limit = np.clip(utilization * rng.normal(1, 0.15, n), 0, 2)
    income_proxy     = rng.lognormal(10.5, 0.5, n) / 1e5

    logit = (
        2.2 * utilization
        - 2.6 * payment_ratio
        + 0.55 * bills_paid_late
        - 0.012 * months_on_book
        + 0.35 * inquiries_6m
        + 1.1 * balance_to_limit * (bills_paid_late > 0)   # interaction
        - 0.8 * income_proxy
        - 0.9
    )
    p_default = 1 / (1 + np.exp(-logit))
    y = (rng.random(n) < p_default).astype(int)

    X = np.column_stack([
        utilization, payment_ratio, bills_paid_late, months_on_book,
        n_credit_lines, inquiries_6m, balance_to_limit, income_proxy,
    ])
    names = [
        "UTILIZATION", "PAYMENT_RATIO", "BILLS_PAID_LATE", "MONTHS_ON_BOOK",
        "N_CREDIT_LINES", "INQUIRIES_6M", "BALANCE_TO_LIMIT", "INCOME_PROXY",
    ]
    return X, y, names

X, y, FEATURES = make_credit_data()
print(f"{X.shape[0]:,} rows, {X.shape[1]} features, default rate {y.mean():.1%}")

30,000 rows, 8 features, default rate 19.9%


In [2]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from compileml.compile import train_whitebox
from compileml.bands import monotone_quantile_bands
from compileml.artifact import build_artifact

teacher = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
).fit(X, y)
whitebox, _ = train_whitebox(X, teacher.predict_proba(X)[:, 1],
                             n_estimators=80, random_state=42)
latent = np.clip(whitebox.predict(X), 0, 1)
artifact = build_artifact(
    whitebox, FEATURES, np.median(X, axis=0),
    monotone_quantile_bands(latent, y, n_bands=8),
    calibration_latent=latent, calibration_y=y,
)
print("artifact", artifact["artifact_hash"][:16], "…")

artifact 443ad995fd31b083 …


C:\Users\crort\AppData\Local\Temp\ipykernel_22188\1248818419.py:13: UserWarning: reason dictionary covers 0/8 features (0%). Uncovered features fall back to generic messages unsuitable for consumer-facing notices: ['UTILIZATION', 'PAYMENT_RATIO', 'BILLS_PAID_LATE', 'MONTHS_ON_BOOK', 'N_CREDIT_LINES', 'INQUIRIES_6M', 'BALANCE_TO_LIMIT', 'INCOME_PROXY']
  artifact = build_artifact(


## SQL: generate, execute, diff against the runtime

In [3]:
import sqlite3
from compileml.export import export_sql
from compileml.runtime import decide

sql = export_sql(artifact, table="applicants", dialect="sqlite")
print(sql[:600], "…")

-- CompileML decision artifact export
-- artifact_hash: 443ad995fd31b0839c4d152bd909295bfff336a89796fcc200a7687d5c9da6c3
-- Integer-exact pipeline: score -> clamp -> display -> band -> PD (ppm).
WITH tree_scores AS (
  SELECT
    src.*,
    (CASE WHEN "BILLS_PAID_LATE" <= 0.5
      THEN CASE WHEN "UTILIZATION" <= 0.6029306948184967
        THEN -22073
        ELSE -2190 END
      ELSE CASE WHEN "UTILIZATION" <= 0.4454369395971298
        THEN -1072
        ELSE 40652 END END) AS tree_0,
    (CASE WHEN "BILLS_PAID_LATE" <= 0.5
      THEN CASE WHEN "BALANCE_TO_LIMIT" <= 0.595427393913269
        …


In [4]:
con = sqlite3.connect(":memory:")
cols = ", ".join(f'"{n}" REAL' for n in FEATURES)
con.execute(f"CREATE TABLE applicants ({cols})")
rows = [[float(v) for v in r] for r in X[:2000]]
con.executemany(f"INSERT INTO applicants VALUES ({','.join('?'*len(FEATURES))})", rows)

cursor = con.execute(sql)
names = [d[0] for d in cursor.description]
mismatches = 0
for row, sql_row in zip(rows, cursor.fetchall()):
    ref = decide(artifact, row, explain=False)
    rec = dict(zip(names, sql_row))
    if (int(rec["latent_int"]), str(rec["band"]), int(rec["pd_ppm"])) != (
        ref["latent_int"], ref["band"], ref["pd_ppm"]
    ):
        mismatches += 1
print(f"rows compared: {len(rows):,}   integer mismatches: {mismatches}")
assert mismatches == 0

rows compared: 2,000   integer mismatches: 0


Zero mismatches — not "close", *equal*. The SQL engine and the Python runtime
are summing the same integers.

## COBOL: what the mainframe team receives

In [5]:
from compileml.export import export_cobol

cobol = export_cobol(artifact, program_id="CMLSCORE")
print(cobol[:1800])
print(f"…\n[{len(cobol.splitlines()):,} lines total]")

       >>SOURCE FORMAT FREE
IDENTIFICATION DIVISION.
PROGRAM-ID. CMLSCORE.
*> ------------------------------------------------------------
*> CompileML decision artifact export (score + band).
*> artifact_hash: 443ad995fd31b0839c4d152bd909295bfff336a89796fcc200a7687d5c9da6c3
*> micro_scale: 1000000   display scale: 1000
*> Integer-exact: leaf values below are the artifact's integers.
*> ------------------------------------------------------------
DATA DIVISION.
WORKING-STORAGE SECTION.
01 F-ACCUM-MICRO      PIC S9(15) COMP-5 VALUE 0.
01 F-LATENT-MICRO     PIC S9(15) COMP-5 VALUE 0.
01 F-LATENT-INT       PIC S9(9)  COMP-5 VALUE 0.
01 FINAL-BAND         PIC X(3)   VALUE SPACES.
01 FEATURE-INPUTS.
   05 F-UTILIZATION                COMP-2 VALUE 0.  *> UTILIZATION
   05 F-PAYMENT-RATIO              COMP-2 VALUE 0.  *> PAYMENT_RATIO
   05 F-BILLS-PAID-LATE            COMP-2 VALUE 0.  *> BILLS_PAID_LATE
   05 F-MONTHS-ON-BOOK             COMP-2 VALUE 0.  *> MONTHS_ON_BOOK
   05 F-N-CREDIT-LI

Notes for the integration team:

- leaf updates are the artifact's integers **verbatim** (`ADD … TO F-ACCUM-MICRO`)
  — there is nothing left to round, so COBOL cannot disagree with Python;
- the band ladder uses strict `<` on integer edges (identical to `bisect_right`);
- feature inputs are `COMP-2` (IEEE binary64); for decimal-arithmetic targets,
  build the artifact with `threshold_decimals=…` so every runtime compares the
  same quantized thresholds;
- model updates on the mainframe = swap the generated program, verify the hash
  in the header comment.